ThinCurr Python Example: Compute eigenstates of  coupled wall, plasma, and coil model
==========
In this example we demonstrate how to compute eigenvalues and eigenvectors for the coupled wall, plasma, and coil RWM model.

**Note:** Running this example requires the [h5py](https://www.h5py.org/) and [pyvista](https://pyvista.org/) python packages, which are installable using `pip` or other standard methods.

In [1]:
%config HistoryManager.enabled = False

In [2]:
import struct
import sys
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
plt.rcParams['figure.figsize']=(6,6)
plt.rcParams['font.weight']='bold'
plt.rcParams['axes.labelweight']='bold'
plt.rcParams['lines.linewidth']=2
plt.rcParams['lines.markeredgewidth']=2
%matplotlib inline
%config InlineBackend.figure_format = "retina"



## Load ThinCurr library

To load the ThinCurr python module we need to tell python where to the module is located. This can be done either through the `PYTHONPATH` environment variable or within a script using `sys.path.append()` as below, where we look for the environement variable `OFT_ROOTPATH` to provide the path to where the OpenFUSIONToolkit is installed (`/Applications/OFT` for binaries on macOS).

In [3]:
thincurr_python_path = '/home/clair/repos/install_release'
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.meshing import build_torus_bnorm_grid, ThinCurr_periodic_toroid
from OpenFUSIONToolkit.ThinCurr.sensor import flux_loop, Mirnov, save_sensors
from OpenFUSIONToolkit.ThinCurr.boozerRWM_d3d import RWMmodel 
from OpenFUSIONToolkit.io import histfile

try:
    import pyvista
    pyvista.set_jupyter_backend('static') # Comment to enable interactive PyVista plots
    have_pyvista = True
except ImportError:
    have_pyvista = False

## Compute eigenvalues

### Setup ThinCurr model
We now create a \ref OpenFUSIONToolkit.OFT_env "OFT_env" instance for execution using two threads and a \ref OpenFUSIONToolkit.ThinCurr.ThinCurr "ThinCurr" instance that utilizes that execution environment.


In [4]:
myOFT = OFT_env(nthreads=2)


#----------------------------------------------
Open FUSION Toolkit Initialized
Development branch:    main
Revision id:           1a4bc3c
Parallelization Info:
  Not compiled with MPI
  # of OpenMP threads =    2
Integer Precisions    =    4   8
Float Precisions      =    4   8  10
Complex Precisions    =    4   8
LA backend            = native
#----------------------------------------------



Once created, we setup the wall (torus) model from an existing HDF5 and XML mesh definition using \ref OpenFUSIONToolkit.ThinCurr.ThinCurr.setup_model "setup_model()".

Finally, we initialize I/O for this model using \ref OpenFUSIONToolkit.ThinCurr.ThinCurr.setup_io "setup_io()" to enable output of plotting files for 3D visualization in [VisIt](https://visit-dav.github.io/visit-website/index.html), [Paraview](https://www.paraview.org/), or using [pyvista](https://pyvista.org/).

In [5]:
tw_torus = ThinCurr(myOFT)
tw_torus.setup_model(mesh_file='DIIID-9_18_24.h5',xml_filename='D3D_coils.xml')
tw_torus.setup_io()



Creating thin-wall model
  Loading V(t) driver coils
    Masked      0 coils from sensors
  No I(t) driver coils found
  Building holes

  Loading region surface resistivity:
     1  1.3200E-05

  Setup complete:
    # of points    =        15271
    # of edges     =        44203
    # of cells     =        28806
    # of holes     =          127
    # of closures  =            0
    # of Vcoils    =           18
    # of Icoils    =            0


### Setup plasma mode driver model
We now create a second \ref OpenFUSIONToolkit.ThinCurr.ThinCurr "ThinCurr" instance for the plasma mode utilizes the same execution environment as above. For this case we also load the plasma mode current patterns created in \ref doc_tCurr_torus_mode.

In [6]:
tw_mode = ThinCurr(myOFT)
tw_mode.setup_model(mesh_file='d3d_mode.h5', xml_filename='D3D_coils.xml')

with h5py.File('d3d_mode.h5', 'r+') as h5_file:
    mode_drive = np.asarray(h5_file['thincurr/driver'])


Creating thin-wall model
  Loading V(t) driver coils
    Masked      0 coils from sensors
  No I(t) driver coils found
  Building holes

  Loading region surface resistivity:
     1  1.3200E-05

  Setup complete:
    # of points    =         3200
    # of edges     =         9600
    # of cells     =         6400
    # of holes     =            2
    # of closures  =            0
    # of Vcoils    =           18
    # of Icoils    =            0


## Calculate constant matrices to establish RWM model
Calculate Uncoupled inductance and resistance matrices for the plasma, wall, and coils. Rmat (called R) is automatically calculated when the rwm model is created. 

In [7]:
def load_sensors(filename):
    '''Read a ThinCurr floops.loc file into a list of flux_loop objects'''
    with open(filename) as f:
        lines = [l.rstrip('\n') for l in f]

    nsensors = int(lines[0].split()[0])
    sensors = []
    i = 1
    for _ in range(nsensors):
        while lines[i].strip() == '':          # skip the blank line before each sensor
            i += 1
        npts_str, scale_str, name = lines[i].split(None, 2)
        npts  = int(npts_str)
        scale = float(scale_str)
        name  = name.strip()
        i += 1
        pts = np.loadtxt(lines[i:i+npts])      # npts rows of x, y, z
        i += npts
        sensors.append(flux_loop(pts, name=name, scale=scale))
    return sensors

sensors = load_sensors('D3D_sensors.loc')

In [8]:
rwm_model = RWMmodel(tw_torus,tw_mode,mode_drive, sensors)

Building coil<->element inductance matrices
  Time = 14s          
Building coil<->coil inductance matrix
  Time = 29s          
  Vcoil    1: L [H] =   1.2018E-04
  Vcoil    2: L [H] =   1.1597E-04
  Vcoil    3: L [H] =   1.2209E-04
  Vcoil    4: L [H] =   1.1456E-04
  Vcoil    5: L [H] =   1.1867E-04
  Vcoil    6: L [H] =   1.1443E-04
  Vcoil    7: L [H] =   4.9649E-06
  Vcoil    8: L [H] =   4.9649E-06
  Vcoil    9: L [H] =   4.9649E-06
  Vcoil   10: L [H] =   4.9653E-06
  Vcoil   11: L [H] =   4.9650E-06
  Vcoil   12: L [H] =   4.9649E-06
  Vcoil   13: L [H] =   4.9643E-06
  Vcoil   14: L [H] =   4.9643E-06
  Vcoil   15: L [H] =   4.9643E-06
  Vcoil   16: L [H] =   4.9657E-06
  Vcoil   17: L [H] =   4.9643E-06
  Vcoil   18: L [H] =   4.9643E-06
Building element<->element self inductance matrix
  Time = 32s          
Building coil<->element inductance matrices
  Time =  1m 11s      
Building coil<->coil inductance matrix
  Time = 28s          
  Vcoil    1: L [H] =   1.2018E-04
  Vc

ValueError: resize only works on single-segment arrays

## Choose s and α Boozer stability and troque parameters and cacluclate inductance 

These values are the primary user inputs into a given RWM model. The s-paramter determines the open-loop stability of the plasma, where a negative s is stable and a positive s is unstable, unless a non-zero α value is provided. The torque parameter α representats a stabilizing plasma rotation. 

Once the s and α parameters have been chosen, the effective coupled inductance matrix can be calculated. 


In [ ]:
rwm_model.Mdw.shape

In [ ]:
s = -0.3
a = 0.5
rwm_model.compute_L_ef(s,a)

## Calculate plasma-response

Perform a s-$\alpha$ scan of frequency-response calculations to try and reproduce something like figure 3 in [Battey (2023)](https://iopscience.iop.org/article/10.1088/1741-4326/accd81)

In [ ]:
a_vals = np.linspace(0.0,0.6,15)
s_vals = np.linspace(-0.1,0.1,20)
coil_vec = np.r_[1.0,1.0j,-1.0,-1.0j]

amp = np.zeros((len(s_vals), len(a_vals))) 
phase = np.zeros((len(s_vals), len(a_vals)))
result = rwm_model.compute_fr(omega=np.pi*2.0*20.0,coil_vec=coil_vec,wall_only=True)
vac_sensor = rwm_model.Msw @ result[:-2] + rwm_model.Msc @ coil_vec

for i, s in enumerate(s_vals):
    print(s)
    for j, a in enumerate(a_vals):
        rwm_model.compute_L_ef(s, a)
        result = rwm_model.compute_fr(omega=np.pi*2.0*20.0,coil_vec=coil_vec)
        sensor_flux = (rwm_model.Msw_ef @ result[:-2] + rwm_model.Msc_ef @ coil_vec + rwm_model.Msd_ef @ result[-2:]) - vac_sensor
        amp[i,j] = np.linalg.norm(sensor_flux)
        phase[i,j] = np.angle(sensor_flux[0])

In [ ]:
S, A = np.meshgrid(s_vals, a_vals, indexing='ij')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

plt.rc('axes', labelsize=16)
plt.rc('figure', titlesize = 20)
pcm1 = ax1.contourf(S, A, amp[:,:], cmap='jet', levels=20)#, vmax=8)
ax1.set_xlabel('s')
ax1.set_ylabel(r'$\alpha$')
ax1.set_title(f'Largest real part')
fig.colorbar(pcm1, ax=ax1)
pcm2 = ax2.contourf(S, A, phase[:,:], cmap='jet', levels=20)
fig.colorbar(pcm2, ax=ax2)

## Calculate eigenvalues and eigenvectors of $$-L^{-1}*R$$

Number of eigenvalues calculated can be sepecified, automatically set to 20. 

In [ ]:
eig_vals, eig_vecs = rwm_model.eigenvalues()

## Plot eigenvalues 
Plot real and imaginary components, where the real part represents the growth or decay rate of the mode, and the imaginary part represents the frequency of roation. 

In [ ]:
rwm_model.plot()

## Perform a parameter sweep 

View the stability of the mode for a range of s and α values. 

In [ ]:
a_vals = np.linspace(0.01,0.2,10)
s_vals = np.linspace(-0.1,0.1,10)

eig = np.zeros((len(s_vals), len(a_vals)), dtype=complex) 
cur = np.zeros((len(s_vals), len(a_vals)), dtype=complex) 


for i, s in enumerate(s_vals):
    for j, a in enumerate(a_vals):
        rwm_model.compute_L_ef(s, a)
        eig_vals, eig_vecs = rwm_model.eigenvalues(20)
        eig_vals = sorted(eig_vals, key=lambda x: abs(x.imag), reverse=False)
        eig_vals = eig_vals[19]
        eig[i,j] = eig_vals
            

Plot the real part of the eigenvalue with the largest imaginary part, where negative values are stable. 

In [ ]:
plt.close('all')

S, A = np.meshgrid(s_vals, a_vals, indexing='ij')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

plt.rc('axes', labelsize=16)
plt.rc('figure', titlesize = 20)
pcm1 = ax1.pcolormesh(S, A, eig.real, shading='auto', cmap='jet')
ax1.set_xlabel('s')
ax1.set_ylabel(r'$\alpha$')
ax1.set_title(f'Largest real part')
fig.colorbar(pcm1, ax=ax1)
pcm1 = ax2.pcolormesh(S, A, eig.imag, shading='auto', cmap='jet')

#%config InlineBackend.figure_format = 'svg'

plt.tight_layout()
plt.show()
